In [4]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras import layers, models, optimizers
import matplotlib.pyplot as plt
import os
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import ModelCheckpoint
import numpy as np

# Parameters
img_size = 224
batch_size = 32
num_classes = 10  # Update this based on your classes
k_folds = 5  # Number of folds for cross-validation

# ImageDataGenerator for training data with augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v3.preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

# ImageDataGenerator for validation and test data
val_test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v3.preprocess_input
)

# Load training, validation, and test data
train_generator = train_datagen.flow_from_directory(
    'GrowQuest_Data_Final/train',  # Your training directory
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_directory(
    'GrowQuest_Data_Final/val',  # Your validation directory
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    'GrowQuest_Data_Final/test',  # Your test directory
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

# Prepare the data (use the first batch for obtaining input shape)
X_data = np.array(train_generator[0][0])  # All training images
y_data = np.array(train_generator[0][1])  # All labels

# Stratified K-fold cross-validation
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, np.argmax(y_data, axis=1))):
    print(f"Training fold {fold+1}/{k_folds}...")

    # Split the data
    X_train, X_val = X_data[train_idx], X_data[val_idx]
    y_train, y_val = y_data[train_idx], y_data[val_idx]

    # --- Base Model ---
    base_model = MobileNetV3Large(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    output = Dense(train_generator.num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=output)

    # Freeze base model
    base_model.trainable = False

    # Compile model
    model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

    # Early stopping to avoid overfitting
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    # Train the model on the current fold
    model.fit(X_train, y_train, epochs=30, batch_size=batch_size, validation_data=(X_val, y_val), callbacks=[early_stop])

    # Evaluate model performance on the validation set
    val_loss, val_acc = model.evaluate(X_val, y_val)
    print(f"Fold {fold+1} - Validation accuracy: {val_acc:.4f}")

    # Optionally, evaluate on the test set after all folds
    test_loss, test_acc = model.evaluate(test_generator)
    print(f"Test accuracy after fold {fold+1}: {test_acc:.4f}")


Found 4199 images belonging to 3 classes.
Found 900 images belonging to 3 classes.
Found 900 images belonging to 3 classes.
Training fold 1/5...
Epoch 1/30
1/1 [==============================] - 5s 5s/step - loss: 3.5851 - accuracy: 0.4800 - val_loss: 3.4320 - val_accuracy: 0.1429
Epoch 2/30
1/1 [==============================] - 0s 258ms/step - loss: 4.0189 - accuracy: 0.4000 - val_loss: 3.2949 - val_accuracy: 0.4286
Epoch 3/30
1/1 [==============================] - 0s 125ms/step - loss: 3.3092 - accuracy: 0.6000 - val_loss: 3.2178 - val_accuracy: 0.5714
Epoch 4/30
1/1 [==============================] - 0s 121ms/step - loss: 3.4652 - accuracy: 0.5600 - val_loss: 3.1350 - val_accuracy: 0.5714
Epoch 5/30
1/1 [==============================] - 0s 113ms/step - loss: 3.0643 - accuracy: 0.6800 - val_loss: 3.0527 - val_accuracy: 0.5714
Epoch 6/30
1/1 [==============================] - 0s 111ms/step - loss: 2.7720 - accuracy: 0.6800 - val_loss: 2.9925 - val_accuracy: 0.8571
Epoch 7/30
1/1 [==

KeyboardInterrupt: 